# How TMDB Movie VAD + Emotion Scores Are Built

This notebook documents the scoring pipeline that produced the **[TMDB Movie VAD + Emotion Intensity Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)** dataset.

**Input:** TMDB Movies Dataset (~1.38M movies, plot keywords)  
**Output:** 3 keyword group columns + 12 affective scores per movie derived from NRC lexicons

### Pipeline

```
TMDB keywords (comma-separated phrases)
  → tokenize on whitespace / hyphens
  → lemmatize (NLTK WordNetLemmatizer, noun / verb / adj POS)
  → look up each lemma in NRC VAD + Emotion Intensity dicts
  → average matched token scores per keyword
  → average keyword scores per movie
  → bucket keywords by valence (positive / negative / unknown)
```

### Lexicons used (Saif M. Mohammad, NRC Canada)

| Lexicon | Words | Output |
|---------|-------|--------|
| NRC Emotion Intensity | 5,891 | continuous 0–1 per emotion |
| NRC VAD | 19,971 | valence, arousal, dominance 0–1 |

In [ ]:
import re
import pandas as pd
from pathlib import Path
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Load TMDB
tmdb_csv = sorted(Path('/kaggle/input').rglob('*.csv'))[0]
df = pd.read_csv(tmdb_csv, low_memory=False, nrows=50_000)
print(f'Loaded {len(df):,} rows — columns: {list(df.columns[:6])} ...')

## Step 1 — Tokenize and lemmatize keywords

Keywords are comma-separated phrases. We split on whitespace/hyphens and try noun, verb, and adjective lemmas in order to maximise lexicon coverage.

In [ ]:
_lemmatizer = WordNetLemmatizer()

def _tokenize(keyword: str) -> list:
    """Split keyword phrase into alpha tokens."""
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]

def _lemma_candidates(tok: str) -> list:
    """Return lemma variants (noun, verb, adj) deduplicated."""
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma); out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

# Example
kw = 'dying and death'
print(f'Keyword: "{kw}"')
print(f'Tokens:  {_tokenize(kw)}')
for tok in _tokenize(kw):
    print(f'  {tok!r} → lemmas: {_lemma_candidates(tok)}')

## Step 2 — NRC lexicon lookup

For each token we walk through its lemma candidates and take the first match in each lexicon. VAD uses NaN (not 0) when a word is missing — 0.5 is the neutral midpoint, so 0 would be falsely negative.

In [ ]:
# Minimal inline lexicons for demonstration (real pipeline loads from files)
# NRC VAD format: word → {valence, arousal, dominance}
sample_vad = {
    'die':    {'valence': 0.2, 'arousal': 0.6, 'dominance': 0.3},
    'death':  {'valence': 0.2, 'arousal': 0.5, 'dominance': 0.3},
    'love':   {'valence': 0.9, 'arousal': 0.7, 'dominance': 0.7},
    'child':  {'valence': 0.8, 'arousal': 0.5, 'dominance': 0.4},
    'fear':   {'valence': 0.2, 'arousal': 0.8, 'dominance': 0.2},
    'ghost':  {'valence': 0.3, 'arousal': 0.6, 'dominance': 0.3},
    'trust':  {'valence': 0.8, 'arousal': 0.4, 'dominance': 0.7},
}

def score_keyword(keyword, vad_lookup):
    tokens = _tokenize(keyword)
    v_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in vad_lookup:
                v_rows.append(vad_lookup[lemma]); break
    if not v_rows:
        return {'valence': float('nan'), 'arousal': float('nan'), 'dominance': float('nan')}
    return {d: sum(r[d] for r in v_rows) / len(v_rows) for d in ('valence', 'arousal', 'dominance')}

# Demo
for kw in ['dying and death', 'child', 'ghost', 'philadelphia']:
    score = score_keyword(kw, sample_vad)
    v = score['valence']
    bucket = 'positive' if v >= 0.5 else ('negative' if v == v else 'unknown')
    print(f'{kw:25s}  valence={v:.2f}  → {bucket}')

## Step 3 — Aggregate to movie level and bucket keywords

For each movie: average keyword scores → movie score. Then bucket each keyword by its individual valence.

In [ ]:
sixth_sense_keywords = 'dying and death, child abuse, sense of guilt, afterlife, confidence, psychology, ghost, child, trust'

kws = [k.strip() for k in sixth_sense_keywords.split(',')]
pos, neg, unk = [], [], []
for kw in kws:
    score = score_keyword(kw, sample_vad)
    v = score['valence']
    if v != v:      unk.append(kw)
    elif v >= 0.5:  pos.append(kw)
    else:           neg.append(kw)

all_scores = [score_keyword(kw, sample_vad) for kw in kws]
vad_vals = [s['valence'] for s in all_scores if s['valence'] == s['valence']]
movie_valence = sum(vad_vals) / len(vad_vals) if vad_vals else float('nan')

print('The Sixth Sense (demo with sample lexicon)')
print(f'  Movie valence:      {movie_valence:.3f}  → {"positive" if movie_valence >= 0.5 else "negative"}')
print(f'  positive_keywords:  {pos}')
print(f'  negative_keywords:  {neg}')
print(f'  unknown_keywords:   {unk}')

## Step 4 — Scale

The real pipeline pre-scores all unique keywords once, then does a dict lookup per movie — ~408K movies/sec on 1.38M rows.

In [ ]:
# Coverage stats on this sample
has_kw = df['keywords'].notna() & (df['keywords'].str.strip() != '')
print(f'Movies with keywords: {has_kw.sum():,} / {len(df):,} ({has_kw.mean():.1%})')

# Unique keywords
all_kws = {kw.strip() for s in df['keywords'].dropna() for kw in s.split(',') if kw.strip()}
print(f'Unique keywords in sample: {len(all_kws):,}')
print(f'\nSample keywords: {sorted(all_kws)[:10]}')

## Full dataset

The complete scored dataset (1.38M movies, 39 columns) is available at:  
**[kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)**

Source code: [github.com/bdelanghe/imdb-kaggle](https://github.com/bdelanghe/imdb-kaggle)

**Lexicon citation:** Mohammad, S.M. (2020). [Practical and Ethical Considerations in the Effective use of Emotion and Sentiment Lexicons](https://www.saifmohammad.com/WebDocs/EmoLex-Ethics-Data-Statement.pdf). NRC Canada.  
License: CC BY-NC-SA 4.0